# Biohub Cell Tracking | Zarr Metadata and Memory Planner

This notebook reads only Zarr JSON metadata from the competition input, then estimates logical array size,
chunk size and safe working-buffer budgets. It does not load image chunks or GEFF graphs.

The goal is a practical memory plan before writing tracking code.

**Verified on all 199 training image stores, 2026-09-07:** each has shape 100 x 64 x 256 x 256 and dtype uint16.

| Working set | Logical memory |
|---|---:|
| One raw T-frame | 8 MiB |
| Raw frame plus two float32 scratch frames | 40 MiB |
| Full raw sequence | 800 MiB |

Thus the listed frame-buffer scenario is 20x smaller than a full raw sequence, before tracker/codec overhead.
The 199 corresponding GEFF stores are annotations, not another 199 independent image sequences.
[Competition/data](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/data)
and [rules](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/rules).
Competition data is listed as CC0. No image pixels or graph nodes are exported. Prepared with AI assistance.

In [ ]:
import json
import os
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    raise FileNotFoundError('Biohub data root not found')

multi_root = os.environ.get('MULTI_DATA_ROOT')
data_root = first_existing([
    os.environ.get('BIOHUB_DATA_ROOT'),
    Path(multi_root) / 'biohub' if multi_root else None,
    Path('/kaggle/input/biohub-cell-tracking-during-development'),
    Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
    Path('work/multi_competition/data/biohub'),
])
working = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('work/multi_competition/local_runs/biohub')
working.mkdir(parents=True, exist_ok=True)
print(f'Data root: {data_root}')

In [ ]:
dtype_sizes = {'uint8': 1, 'int8': 1, 'uint16': 2, 'int16': 2, 'uint32': 4, 'int32': 4,
               'float32': 4, 'float64': 8}

def digest_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

rows = []
root_files = sorted((data_root / 'train').glob('*.zarr/zarr.json'))
if not root_files:
    root_files = sorted(data_root.glob('train/*.zarr/zarr.json'))
for root_json in root_files:
    store = root_json.parent
    arr_json = store / '0' / 'zarr.json'
    if not arr_json.exists():
        continue
    root_meta = json.loads(root_json.read_text())
    arr_meta = json.loads(arr_json.read_text())
    shape = arr_meta.get('shape') or []
    chunks = arr_meta.get('chunk_grid', {}).get('configuration', {}).get('chunk_shape') or arr_meta.get('chunks') or []
    dtype = str(arr_meta.get('data_type') or arr_meta.get('dtype')).replace('numpy.', '')
    axes = [a.get('name') for a in root_meta.get('attributes', {}).get('multiscales', [{}])[0].get('axes', [])]
    assert [a.upper() for a in axes] == ['T', 'Z', 'Y', 'X'], 'Unsupported axis order'
    itemsize = dtype_sizes.get(dtype, np.dtype(dtype).itemsize if dtype not in (None, 'None') else np.nan)
    raw_bytes = int(np.prod(shape) * itemsize) if shape else 0
    chunk_bytes = int(np.prod(chunks) * itemsize) if chunks else 0
    frame_shape = shape[1:] if len(shape) == 4 else shape
    frame_bytes = int(np.prod(frame_shape) * itemsize) if frame_shape else 0
    rows.append({
        'store_hash': hashlib.sha256(store.name.encode()).hexdigest()[:12],
        'axes': ','.join(axes),
        'shape': 'x'.join(map(str, shape)),
        'chunks': 'x'.join(map(str, chunks)),
        'dtype': dtype,
        'itemsize': itemsize,
        'timepoints': shape[0] if len(shape) == 4 else np.nan,
        'z_slices': shape[1] if len(shape) == 4 else np.nan,
        'height': shape[-2] if len(shape) >= 2 else np.nan,
        'width': shape[-1] if len(shape) >= 1 else np.nan,
        'raw_mib': raw_bytes / 2**20,
        'chunk_mib': chunk_bytes / 2**20,
        'frame_mib': frame_bytes / 2**20,
        'root_json_sha256': digest_file(root_json),
        'array_json_sha256': digest_file(arr_json),
    })

meta_df = pd.DataFrame(rows)
assert len(meta_df) > 0, 'No readable metadata'
print(f'Zarr stores with readable root+array metadata: {len(meta_df):,}')
meta_df.head()

In [ ]:
summary = {
    'competition': 'biohub-cell-tracking-during-development',
    'metadata_only': True,
    'zarr_stores_observed': int(len(meta_df)),
    'dtype_counts': meta_df['dtype'].value_counts().to_dict(),
    'shape_counts': meta_df['shape'].value_counts().head(10).to_dict(),
    'median_raw_mib': float(meta_df['raw_mib'].median()),
    'max_raw_mib': float(meta_df['raw_mib'].max()),
    'median_chunk_mib': float(meta_df['chunk_mib'].median()),
    'median_frame_mib': float(meta_df['frame_mib'].median()),
}
print(json.dumps(summary, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
meta_df['raw_mib'].plot(kind='hist', bins=min(20, max(3, len(meta_df)//2)), ax=axes[0], color='#2d6f8f')
axes[0].set_title('Logical array size')
axes[0].set_xlabel('MiB, uncompressed')

meta_df['chunk_mib'].plot(kind='hist', bins=min(20, max(3, len(meta_df)//2)), ax=axes[1], color='#875c8d')
axes[1].set_title('Single chunk size')
axes[1].set_xlabel('MiB, uncompressed')

axes[2].scatter(meta_df['timepoints'], meta_df['frame_mib'], color='#a85b36', alpha=0.75)
axes[2].set_title('Timepoints vs one T-frame')
axes[2].set_xlabel('T')
axes[2].set_ylabel('MiB per frame')
fig.tight_layout()
fig.savefig(working / 'biohub_memory_planner.png', dpi=160)
plt.show()

In [ ]:
scenarios = []
for _, row in meta_df.iterrows():
    frame = row['frame_mib']
    raw = row['raw_mib']
    scenarios.append({'store_hash': row['store_hash'], 'scenario': 'one_raw_frame', 'estimated_mib': frame})
    scenarios.append({'store_hash': row['store_hash'], 'scenario': 'raw_plus_two_float32_frames', 'estimated_mib': frame + 2 * frame * 4 / row['itemsize']})
    scenarios.append({'store_hash': row['store_hash'], 'scenario': 'full_logical_array', 'estimated_mib': raw})
scenario_df = pd.DataFrame(scenarios)
scenario_summary = scenario_df.groupby('scenario')['estimated_mib'].agg(['min', 'median', 'max']).reset_index()
scenario_summary.to_csv(working / 'biohub_memory_scenarios.csv', index=False)
(working / 'summary.json').write_text(json.dumps(summary, indent=2))
scenario_summary

## Practical use

Tracking pipelines should be designed around chunk and frame budgets first. The full logical array size is not
the right working-set target for a CPU baseline; frame streaming plus small float32 scratch buffers is the safer
default until a specific tracker proves it needs larger temporal windows.

Sizes are uncompressed logical bytes, not download size or measured process/GPU memory. Codec buffers,
tracker state, masks and allocator overhead are excluded. Local checks use 12 metadata stores;
the Kaggle run scans all available training image stores, not GEFF graph stores.